# vLLM Paper Reproduction (CSE 231)

Reproducing Kwon et al., *Efficient Memory Management for Large Language Model Serving with PagedAttention* (SOSP '23) on two T4 GPUs using `facebook/opt-2.7b`. Targets Figures 12, 16, and 18b. 

## 1. Install dependencies

vLLM 0.6.4 pins a specific torch version, so the preinstalled torch stack is removed first. Restart the kernel after this cell.

In [ ]:
!pip uninstall -qy torch torchvision torchaudio

!pip install -q --upgrade pip
!pip install -q \
    vllm==0.6.4.post1 \
    transformers==4.45.2 \
    huggingface_hub==0.25.2 \
    numpy==1.26.4 \
    pandas matplotlib

## 2. Environment, dataset, model cache

GPU check, ShareGPT download (~700 MB), `facebook/opt-2.7b` cache (~5.4 GB safetensors).

In [ ]:
import sys, urllib.request
from pathlib import Path

DATA_DIR = Path("data")
RESULTS_DIR = Path("results")
SHAREGPT_PATH = DATA_DIR / "sharegpt.json"
SHAREGPT_URL = (
    "https://huggingface.co/datasets/anon8231489123/ShareGPT_Vicuna_unfiltered"
    "/resolve/main/ShareGPT_V3_unfiltered_cleaned_split.json"
)
MODEL_ID = "facebook/opt-2.7b"

import torch
print(f"python = {sys.version.split()[0]}")
print(f"torch  = {torch.__version__}  (cuda={torch.version.cuda})")
assert torch.cuda.is_available(), "CUDA not available."
dev = torch.cuda.get_device_properties(0)
free, total = torch.cuda.mem_get_info(0)
print(f"gpu    = {dev.name}  sm={dev.major}.{dev.minor}  mem={total/1e9:.1f} GB  free={free/1e9:.1f} GB")

def download_sharegpt(path: Path) -> None:
    if path.exists():
        print(f"sharegpt = {path} ({path.stat().st_size/1e6:.1f} MB, cached)")
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".part")
    print(f"sharegpt = downloading to {path}")
    with urllib.request.urlopen(SHAREGPT_URL) as resp, open(tmp, "wb") as fh:
        total_bytes = int(resp.headers.get("Content-Length", 0))
        downloaded, next_pct = 0, 5
        while True:
            buf = resp.read(1 << 20)
            if not buf:
                break
            fh.write(buf)
            downloaded += len(buf)
            if total_bytes and 100 * downloaded / total_bytes >= next_pct:
                pct = int(100 * downloaded / total_bytes)
                print(f"           {pct:3d}%  ({downloaded/1e6:6.0f} / {total_bytes/1e6:6.0f} MB)", flush=True)
                next_pct = pct + 5
    tmp.rename(path)
    print(f"sharegpt = done ({path.stat().st_size/1e6:.1f} MB)")

download_sharegpt(SHAREGPT_PATH)

from huggingface_hub import snapshot_download
print(f"caching {MODEL_ID}")
MODEL_DIR = snapshot_download(
    repo_id=MODEL_ID,
    allow_patterns=["*.json", "*.txt", "tokenizer*", "*.safetensors", "merges.txt", "vocab.json"],
)
print(f"model cached at {MODEL_DIR}")

RESULTS_DIR.mkdir(exist_ok=True)

## 3. Shared helpers

ShareGPT filter (≥2 turns, prompt ≥ 4 tokens, prompt + completion ≤ 1024 tokens), tokenizer, sweep parameters.

In [ ]:
import json, random
import numpy as np
from transformers import AutoTokenizer

SMOKE_TEST = False

SEED = 0
NUM_REQUESTS_FULL  = 100
NUM_REQUESTS_SMOKE = 20
RATES_FULL  = [0.5, 1.0, 2.0, 4.0, 8.0, 16.0, 32.0]
RATES_SMOKE = [2.0, 8.0]

NUM_REQUESTS = NUM_REQUESTS_SMOKE if SMOKE_TEST else NUM_REQUESTS_FULL
RATES        = RATES_SMOKE        if SMOKE_TEST else RATES_FULL

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

def load_sharegpt_prompts(n: int, seed: int = SEED):
    data = json.loads(SHAREGPT_PATH.read_text())
    rng = random.Random(seed)
    rng.shuffle(data)
    out = []
    for conv in data:
        if len(out) >= n:
            break
        turns = conv.get("conversations", [])
        if len(turns) < 2 or turns[0].get("from") != "human" or turns[1].get("from") != "gpt":
            continue
        prompt_text, target_text = turns[0]["value"], turns[1]["value"]
        prompt_ids = tokenizer(prompt_text).input_ids
        target_ids = tokenizer(target_text).input_ids
        if len(prompt_ids) < 4 or len(prompt_ids) + len(target_ids) > 1024:
            continue
        out.append({
            "prompt": prompt_text,
            "prompt_len": len(prompt_ids),
            "target_len": len(target_ids),
        })
    if len(out) < n:
        raise RuntimeError(f"Only {len(out)} prompts pass filter (wanted {n}).")
    return out

def check_gpu_free(min_fraction: float = 0.75) -> None:
    """Abort early if the GPU is already partially allocated by a prior cell."""
    import torch
    free, total = torch.cuda.mem_get_info(0)
    if free / total < min_fraction:
        raise RuntimeError(
            f"Only {free/1e9:.1f} GiB free of {total/1e9:.1f} GiB ({100*free/total:.0f}%). "
            "Restart the kernel and re-run setup before this cell."
        )

## 4. HuggingFace baseline (Fig 12)

Static padded batching, batch size 8. 

In [ ]:
import gc, time
import torch
from transformers import AutoModelForCausalLM

check_gpu_free()

RESULT_PATH = RESULTS_DIR / "plan_a_hf.json"
OVERWRITE = False
BATCH_SIZE = 8

if RESULT_PATH.exists() and not OVERWRITE:
    print(f"{RESULT_PATH} exists — set OVERWRITE=True to rerun.")
else:
    prompts = load_sharegpt_prompts(NUM_REQUESTS)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16).cuda().eval()

    def sweep_rate_hf(rate):
        rng = np.random.default_rng(SEED)
        deltas = rng.exponential(1.0 / rate, size=len(prompts))
        arrivals = np.cumsum(deltas)

        t0 = time.perf_counter()
        per_req = []
        idx = 0
        while idx < len(prompts):
            end = min(idx + BATCH_SIZE, len(prompts))
            batch = prompts[idx:end]
            last_arrival = arrivals[end - 1]
            gap = last_arrival - (time.perf_counter() - t0)
            if gap > 0:
                time.sleep(gap)

            texts = [p["prompt"] for p in batch]
            inputs = tokenizer(
                texts, return_tensors="pt", padding=True,
                truncation=True, max_length=1024,
            ).to("cuda")
            max_new = max(p["target_len"] for p in batch)
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=max_new,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id,
                )
            t_done = time.perf_counter()
            out_len = int(outputs.shape[1] - inputs.input_ids.shape[1])
            for i in range(len(batch)):
                non_pad = int(inputs.input_ids[i].ne(tokenizer.pad_token_id).sum().item())
                per_req.append({
                    "rid": idx + i,
                    "latency": t_done - (t0 + arrivals[idx + i]),
                    "input_tokens": non_pad,
                    "output_tokens": out_len,
                })
            idx = end

        t_end = time.perf_counter()
        wall = t_end - t0
        lats = np.array([r["latency"] for r in per_req])
        total_out = sum(r["output_tokens"] for r in per_req)
        return {
            "rate": rate,
            "num_requests": len(per_req),
            "wall_time": wall,
            "throughput_req_s": len(per_req) / wall,
            "throughput_tok_s": total_out / wall,
            "mean_latency": float(lats.mean()),
            "p99_latency":  float(np.percentile(lats, 99)),
            "per_request": per_req,
        }

    results = []
    for rate in RATES:
        print(f"\n=== rate = {rate} req/s ===")
        r = sweep_rate_hf(rate)
        print(f"  throughput = {r['throughput_req_s']:.2f} req/s, {r['throughput_tok_s']:.0f} tok/s")
        print(f"  latency    = mean {r['mean_latency']:.2f}s, p99 {r['p99_latency']:.2f}s")
        results.append(r)

    RESULT_PATH.write_text(json.dumps(results, indent=2))
    print(f"\nwrote {RESULT_PATH}")

    del model
    gc.collect(); torch.cuda.empty_cache()

## 5. vLLM (Fig 12)

Async request-rate sweep with Poisson arrivals fed to `AsyncLLMEngine`. 

In [ ]:
import asyncio, gc, time
import torch
from vllm import AsyncEngineArgs, AsyncLLMEngine, SamplingParams

check_gpu_free()

RESULT_PATH = RESULTS_DIR / "plan_a_vllm.json"
OVERWRITE = True   # rerun to extend rate sweep past saturation

if RESULT_PATH.exists() and not OVERWRITE:
    print(f"{RESULT_PATH} exists — set OVERWRITE=True to rerun.")
else:
    prompts = load_sharegpt_prompts(NUM_REQUESTS)
    print(f"prompts = {len(prompts)}  "
          f"(mean prompt_len={np.mean([p['prompt_len'] for p in prompts]):.0f}, "
          f"mean target_len={np.mean([p['target_len'] for p in prompts]):.0f})")

    engine_args = AsyncEngineArgs(
        model=MODEL_ID,
        dtype="float16",
        gpu_memory_utilization=0.9,
        max_model_len=1024,
        enforce_eager=True,
        seed=SEED,
        disable_log_requests=True,
    )
    engine = AsyncLLMEngine.from_engine_args(engine_args)

    async def one_request(rid, spec):
        sp = SamplingParams(temperature=0.0, max_tokens=spec["target_len"], ignore_eos=True)
        t0 = time.perf_counter()
        final = None
        async for out in engine.generate(spec["prompt"], sp, str(rid)):
            final = out
        t1 = time.perf_counter()
        return {
            "rid": rid,
            "latency": t1 - t0,
            "input_tokens": len(final.prompt_token_ids),
            "output_tokens": len(final.outputs[0].token_ids),
        }

    async def sweep_rate(rate):
        rng = np.random.default_rng(SEED)
        deltas = rng.exponential(1.0 / rate, size=len(prompts))
        arrivals = np.cumsum(deltas)

        t_start = time.perf_counter()
        tasks = []
        for i, spec in enumerate(prompts):
            target = t_start + arrivals[i]
            gap = target - time.perf_counter()
            if gap > 0:
                await asyncio.sleep(gap)
            tasks.append(asyncio.create_task(one_request(i, spec)))
        per_req = await asyncio.gather(*tasks)
        t_end = time.perf_counter()

        wall = t_end - t_start
        lats = np.array([r["latency"] for r in per_req])
        total_out = sum(r["output_tokens"] for r in per_req)
        return {
            "rate": rate,
            "num_requests": len(per_req),
            "wall_time": wall,
            "throughput_req_s": len(per_req) / wall,
            "throughput_tok_s": total_out / wall,
            "mean_latency": float(lats.mean()),
            "p50_latency":  float(np.percentile(lats, 50)),
            "p99_latency":  float(np.percentile(lats, 99)),
            "per_request": per_req,
        }

    results = []
    for rate in RATES:
        print(f"\n=== rate = {rate} req/s ===")
        r = await sweep_rate(rate)
        print(f"  throughput = {r['throughput_req_s']:.2f} req/s, {r['throughput_tok_s']:.0f} tok/s")
        print(f"  latency    = mean {r['mean_latency']:.2f}s, p50 {r['p50_latency']:.2f}s, p99 {r['p99_latency']:.2f}s")
        results.append(r)

    RESULT_PATH.write_text(json.dumps(results, indent=2))
    print(f"\nwrote {RESULT_PATH}")

    del engine
    gc.collect(); torch.cuda.empty_cache()

## 6. Block-size ablation (Fig 18b)

Offline sweep over `block_size ∈ {8, 16, 32}`. The xformers PagedAttention kernel on T4 (sm_75) ships templates only for those three values. Restart the kernel before running this cell.

In [ ]:
import gc, time
import torch
from vllm import LLM, SamplingParams

check_gpu_free()

RESULT_PATH = RESULTS_DIR / "plan_a_blocksize.json"
OVERWRITE = True   # rerun to capture per-request latency
BLOCK_SIZES = [16, 32] if SMOKE_TEST else [8, 16, 32]

if RESULT_PATH.exists() and not OVERWRITE:
    print(f"{RESULT_PATH} exists — set OVERWRITE=True to rerun.")
else:
    prompts = load_sharegpt_prompts(NUM_REQUESTS)
    texts = [p["prompt"] for p in prompts]
    sampling_params = [
        SamplingParams(temperature=0.0, max_tokens=p["target_len"], ignore_eos=True)
        for p in prompts
    ]

    results = []
    for bs in BLOCK_SIZES:
        print(f"\n=== block_size = {bs} ===")
        gc.collect(); torch.cuda.empty_cache()
        llm = LLM(
            model=MODEL_ID, dtype="float16", gpu_memory_utilization=0.9,
            max_model_len=1024, block_size=bs, enforce_eager=True, seed=SEED,
        )
        t0 = time.perf_counter()
        outs = llm.generate(texts, sampling_params)
        t1 = time.perf_counter()
        wall = t1 - t0

        per_req = []
        for o in outs:
            m = o.metrics
            if m is not None and getattr(m, "arrival_time", None) is not None and getattr(m, "finished_time", None) is not None:
                latency = m.finished_time - m.arrival_time
            else:
                latency = wall
            out_tokens = len(o.outputs[0].token_ids)
            per_req.append({"latency": latency, "output_tokens": out_tokens})

        nls = [p["latency"] / max(p["output_tokens"], 1) for p in per_req]
        total_out = sum(p["output_tokens"] for p in per_req)
        results.append({
            "block_size": bs,
            "num_requests": len(outs),
            "wall_time": wall,
            "throughput_req_s": len(outs) / wall,
            "throughput_tok_s": total_out / wall,
            "mean_norm_latency": float(np.mean(nls)),
            "p99_norm_latency": float(np.percentile(nls, 99)),
            "per_request": per_req,
        })
        RESULT_PATH.write_text(json.dumps(results, indent=2))
        print(f"  throughput = {len(outs)/wall:.2f} req/s, {total_out/wall:.0f} tok/s, "
              f"mean_norm_lat = {np.mean(nls):.4f} s/tok")
        del llm
        gc.collect(); torch.cuda.empty_cache()

    print(f"\nwrote {RESULT_PATH}")

## 7. Shared-prefix workload (Fig 16)

50 requests sharing a ~500-token prefix with unique short suffixes. Compares `enable_prefix_caching` on vs off. 

In [ ]:
import asyncio, gc, time
import torch
from vllm import AsyncEngineArgs, AsyncLLMEngine, SamplingParams

check_gpu_free()

RESULT_PATH = RESULTS_DIR / "plan_b_prefix.json"
OVERWRITE = False   # start fresh; flip to True to wipe prior data
NUM_REQUESTS_PREFIX = 20 if SMOKE_TEST else 100
OUTPUT_TOKENS = 64
PREFIX_RATES = [2.0, 8.0] if SMOKE_TEST else [1.0, 2.0, 4.0, 8.0, 16.0, 32.0]

prefix = ("You are an expert research assistant. Answer concisely. " * 40)
prefix_len = len(tokenizer(prefix).input_ids)

# Load existing data so a kernel restart between configs picks up where it left off.
if RESULT_PATH.exists() and not OVERWRITE:
    results = json.loads(RESULT_PATH.read_text())
else:
    results = {}
results.setdefault("prefix_off", [])
results.setdefault("prefix_on", [])
results["prefix_tokens"] = int(prefix_len)

todo = []
if len(results["prefix_off"]) < len(PREFIX_RATES):
    todo.append(False)
if len(results["prefix_on"]) < len(PREFIX_RATES):
    todo.append(True)

if not todo:
    print(f"{RESULT_PATH} already has both configs — set OVERWRITE=True to rerun.")
else:
    print(f"prefix = ~{prefix_len} tokens; running configs {[ 'on' if x else 'off' for x in todo ]}")
    prompts = [
        {"prompt": prefix + f" Question {i}: what is the square of {i}?", "target_len": OUTPUT_TOKENS}
        for i in range(NUM_REQUESTS_PREFIX)
    ]

    async def one_request(engine, rid, spec):
        sp = SamplingParams(temperature=0.0, max_tokens=spec["target_len"], ignore_eos=True)
        t0 = time.perf_counter()
        final = None
        async for out in engine.generate(spec["prompt"], sp, str(rid)):
            final = out
        t1 = time.perf_counter()
        return {
            "rid": rid,
            "latency": t1 - t0,
            "input_tokens": len(final.prompt_token_ids),
            "output_tokens": len(final.outputs[0].token_ids),
        }

    async def sweep_rate(engine, rate):
        rng = np.random.default_rng(SEED)
        deltas = rng.exponential(1.0 / rate, size=len(prompts))
        arrivals = np.cumsum(deltas)
        t_start = time.perf_counter()
        tasks = []
        for i, spec in enumerate(prompts):
            target = t_start + arrivals[i]
            gap = target - time.perf_counter()
            if gap > 0:
                await asyncio.sleep(gap)
            tasks.append(asyncio.create_task(one_request(engine, i, spec)))
        per_req = await asyncio.gather(*tasks)
        t_end = time.perf_counter()
        wall = t_end - t_start
        nls = [r["latency"] / max(r["output_tokens"], 1) for r in per_req]
        return {
            "rate": rate,
            "num_requests": len(per_req),
            "wall_time": wall,
            "throughput_req_s": len(per_req) / wall,
            "mean_norm_latency": float(np.mean(nls)),
            "p99_norm_latency": float(np.percentile(nls, 99)),
            "per_request": per_req,
        }

    for enable_pc in todo:
        label = "prefix_on" if enable_pc else "prefix_off"
        already_done = len(results[label])
        remaining = PREFIX_RATES[already_done:]
        if not remaining:
            continue
        print(f"\n=== prefix caching {'ON' if enable_pc else 'OFF'} (rates {remaining}) ===")
        gc.collect(); torch.cuda.empty_cache()
        engine_args = AsyncEngineArgs(
            model=MODEL_ID, dtype="float16", gpu_memory_utilization=0.9,
            max_model_len=1024, enforce_eager=True, seed=SEED,
            disable_log_requests=True, enable_prefix_caching=enable_pc,
        )
        engine = AsyncLLMEngine.from_engine_args(engine_args)
        for rate in remaining:
            print(f"  rate = {rate} req/s")
            r = await sweep_rate(engine, rate)
            print(f"    norm_lat = {r['mean_norm_latency']:.4f} s/tok, throughput = {r['throughput_req_s']:.2f} req/s")
            results[label].append(r)
            RESULT_PATH.write_text(json.dumps(results, indent=2))
        del engine
        gc.collect(); torch.cuda.empty_cache()

    print(f"\nwrote {RESULT_PATH}")

## 8. Plots

In [ ]:
import matplotlib.pyplot as plt

def _load(name):
    p = RESULTS_DIR / name
    return json.loads(p.read_text()) if p.exists() else None

def _norm_lat_per_rate(rate_list):
    out = []
    for d in rate_list:
        per = d.get("per_request", [])
        if per:
            nls = [p["latency"] / max(p["output_tokens"], 1) for p in per]
            out.append((d["rate"], float(sum(nls) / len(nls))))
        elif "mean_norm_latency" in d:
            out.append((d["rate"], d["mean_norm_latency"]))
    return out

fig12_vllm = _load("plan_a_vllm.json")
fig12_hf   = _load("plan_a_hf.json")
fig18b     = _load("plan_a_blocksize.json")
fig16      = _load("plan_b_prefix.json")

# Fig 12: normalized latency vs request rate
if fig12_vllm or fig12_hf:
    plt.figure(figsize=(6, 3.4))
    for label, data, style, color in [
        ("vLLM",      fig12_vllm, "o-",  "tab:blue"),
        ("HF padded", fig12_hf,   "s--", "tab:gray"),
    ]:
        if not data:
            continue
        rows = _norm_lat_per_rate(data)
        plt.plot([r for r, _ in rows], [n for _, n in rows], style, color=color, label=label)
    plt.xlabel("Request rate (req/s)")
    plt.ylabel("Normalized latency (s/token)")
    plt.title("OPT-2.7B, ShareGPT, T4")
    plt.legend(loc="upper left")
    plt.grid(True, alpha=0.3)
    plt.ylim(bottom=0)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "fig12.png", dpi=140)
    plt.show()

# Fig 18b: normalized latency vs block size
if fig18b:
    plt.figure(figsize=(5, 3.2))
    xs  = [d["block_size"] for d in fig18b]
    nls = [d.get("mean_norm_latency", 1.0 / d["throughput_tok_s"]) for d in fig18b]
    plt.plot(xs, nls, "o-", color="tab:blue")
    plt.xscale("log", base=2)
    plt.xticks(xs, [str(x) for x in xs])
    plt.xlabel("Block size")
    plt.ylabel("Normalized latency (s/token)")
    plt.title("OPT-2.7B, ShareGPT, T4")
    plt.grid(True, alpha=0.3)
    plt.ylim(bottom=0)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "fig18b.png", dpi=140)
    plt.show()

# Fig 16: prefix on vs off, normalized latency vs request rate
if fig16 and isinstance(fig16.get("prefix_off"), list):
    plt.figure(figsize=(6, 3.4))
    for label, key, style, color in [
        ("prefix off", "prefix_off", "s--", "tab:gray"),
        ("prefix on",  "prefix_on",  "o-",  "tab:blue"),
    ]:
        rows = _norm_lat_per_rate(fig16.get(key, []))
        if rows:
            plt.plot([r for r, _ in rows], [n for _, n in rows], style, color=color, label=label)
    plt.xlabel("Request rate (req/s)")
    plt.ylabel("Normalized latency (s/token)")
    plt.title("Shared-prefix workload (OPT-2.7B, T4)")
    plt.legend(loc="upper left")
    plt.grid(True, alpha=0.3)
    plt.ylim(bottom=0)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "fig16.png", dpi=140)
    plt.show()